<div style="background: linear-gradient(135deg, #1a0a0a 0%, #2d0a2d 50%, #1a0a1a 100%); padding: 30px; border-radius: 15px; text-align: center; border: 2px solid #8b0000; box-shadow: 0 0 30px rgba(139,0,0,0.5);">
  <h1 style="color: #ff4444; font-size: 2.5em; margin: 0; text-shadow: 0 0 20px #ff0000;">&#127908; Elbek Studiya</h1>
  <h2 style="color: #cc44ff; font-size: 1.8em; margin: 10px 0;">RVC Model Trainer</h2>
  <p style="color: #aaaaaa; font-size: 1.1em;">Powered by <a href="https://github.com/Bebra777228/TrainVocModel-EN" style="color:#ff6666;">Bebra777228/TrainVocModel-EN</a></p>
  <p style="color: #888; font-size: 0.9em;">&#128308; Google Colab T4 GPU bilan ishlaydi &bull; ZIP: .pth + .index</p>
</div>

In [ ]:
#@title <big>&#128308; **1-QADAM: Ornatish (Install)**</big>
#@markdown ### GPU ulangan bolishi shart! Faqat bir marta ishga tushiring.

REPO_BRANCH = "main"
ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"
GITHUB_RAW_URL = f"https://raw.githubusercontent.com/Bebra777228/TrainVocModel-EN/refs/heads/{REPO_BRANCH}"

import os
import sys
import torch
from ipywidgets import Button
from google.colab import drive
from IPython.display import clear_output

# GPU TEKSHIRISH
print("GPU tekshirilmoqda...")
if not torch.cuda.is_available():
    raise Exception("\033[91mGPU topilmadi!\033[0m\nColab sozlamalarida GPU (T4) ni yoqing.")
print(f"\033[92mGPU tayyor: {torch.cuda.get_device_name(0)}\033[0m")

# GOOGLE DRIVE ULASH
if not os.path.isdir("/content/drive"):
    print("\nGoogle Drive ulanmoqda...")
    drive.mount("/content/drive")
if not os.path.exists("/content/dataset"):
    os.makedirs("/content/dataset")

# REPONI CLONE QILISH
if not os.path.isdir(ROOT_DIR):
    print("\nRepo clone qilinmoqda...")
    os.system(f"git clone https://github.com/Bebra777228/TrainVocModel-EN -b {REPO_BRANCH} {ROOT_DIR} &> /dev/null")

os.chdir(ROOT_DIR)
clear_output()

print("Ornatish boshlandi. ~6 daqiqa ketadi...")

# PYTHON 3.10
print("[1/5] - Python 3.10 ga otilmoqda...")
os.system("sudo apt update &> /dev/null")
os.system("sudo apt install python3.10 &> /dev/null")
os.system("sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1 &> /dev/null")
os.system("sudo update-alternatives --set python3 /usr/bin/python3.10 &> /dev/null")
os.system("curl -sS https://bootstrap.pypa.io/get-pip.py | python3 &> /dev/null")
sys.path.append("/usr/local/lib/python3.10/dist-packages")

# SKRIPTLARNI YUKLAB OLISH
print("[2/5] - Zarur fayllar yuklanmoqda...")
files_to_download = [
    (f"{GITHUB_RAW_URL}/requirements.txt",                       f"{ROOT_DIR}/requirements.txt"),
    (f"{GITHUB_RAW_URL}/download_files.py",                      f"{ROOT_DIR}/download_files.py"),
    (f"{GITHUB_RAW_URL}/configs/config.py",                      f"{ROOT_DIR}/configs/config.py"),
    (f"{GITHUB_RAW_URL}/infer/lib/rmvpe.py",                     f"{ROOT_DIR}/infer/lib/rmvpe.py"),
    (f"{GITHUB_RAW_URL}/infer/lib/slicer.py",                    f"{ROOT_DIR}/infer/lib/slicer.py"),
    (f"{GITHUB_RAW_URL}/infer/lib/train/utils.py",               f"{ROOT_DIR}/infer/lib/train/utils.py"),
    (f"{GITHUB_RAW_URL}/infer/modules/train/train.py",           f"{ROOT_DIR}/infer/modules/train/train.py"),
    (f"{GITHUB_RAW_URL}/infer/modules/vc/infer_cli.py",          f"{ROOT_DIR}/infer/modules/vc/infer_cli.py"),
    (f"{GITHUB_RAW_URL}/infer/lib/train/process_ckpt.py",        f"{ROOT_DIR}/infer/lib/train/process_ckpt.py"),
    (f"{GITHUB_RAW_URL}/infer/modules/train/extract_f0.py",      f"{ROOT_DIR}/infer/modules/train/extract_f0.py"),
    (f"{GITHUB_RAW_URL}/infer/modules/train/preprocess.py",      f"{ROOT_DIR}/infer/modules/train/preprocess.py"),
    (f"{GITHUB_RAW_URL}/infer/modules/train/extract_index.py",   f"{ROOT_DIR}/infer/modules/train/extract_index.py"),
    (f"{GITHUB_RAW_URL}/infer/modules/train/extract_feature.py", f"{ROOT_DIR}/infer/modules/train/extract_feature.py"),
]
for url, dest in files_to_download:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    os.system(f"wget {url} -O {dest} &> /dev/null")

# PIP ORNATISH
print("[3/5] - Kutubxonalar ornatilmoqda...")
os.system("pip install --upgrade pip==24.0 &> /dev/null")
os.system("pip install --no-cache-dir -qq -r requirements.txt &> /dev/null")
os.system("sudo apt install -y aria2 ffmpeg &> /dev/null")

# HuBERT MODEL
print("[4/5] - HuBERT modeli yuklanmoqda...")
os.system('python download_files.py "./assets/" "contentvec_base" &> /dev/null')

# KERAKSIZ FAYLLAR
print("[5/5] - Keraksiz fayllar ochirilmoqda...")
os.system("rm -rf /content/sample_data")
os.system(f"rm -rf {ROOT_DIR}/.git")
os.system(f"rm -rf {ROOT_DIR}/notebooks")

clear_output()
print("\033[92m Ornatish muvaffaqiyatli tugadi!\033[0m")
Button(description=" Tayyor!", button_style="success")


In [ ]:
#@title <big>&#128993; **2-QADAM: Malumotlarni qayta ishlash**</big>
#@markdown ### Sozlamalar
model_name = "MyVoice" #@param {type:"string"}
dataset_folder = "/content/dataset" #@param {type:"string"}
sample_rate = "40k" #@param ["32k", "40k", "48k"]
f0_method = "rmvpe" #@param ["harvest", "rmvpe"]
index_algorithm = "Auto" #@param ["Auto", "Faiss", "KMeans"]
percentage = 3.0 #@param {type:"number"}

ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"

import os
from ipywidgets import Button
from IPython.display import clear_output

os.chdir(ROOT_DIR)

# sr_int hisoblash (40k -> 40000)
sr_int = int(sample_rate.rstrip("k")) * 1000

exp_dir = f"./logs/{model_name}"
os.makedirs(exp_dir, exist_ok=True)

preprocess_script    = f"{ROOT_DIR}/infer/modules/train/preprocess.py"
extract_f0_script    = f"{ROOT_DIR}/infer/modules/train/extract_f0.py"
extract_feature_script = f"{ROOT_DIR}/infer/modules/train/extract_feature.py"
extract_index_script   = f"{ROOT_DIR}/infer/modules/train/extract_index.py"

# 1. PREPROCESS
print("[1/4] - Audio qayta ishlanmoqda...")
if os.path.exists(preprocess_script):
    ret = os.system(f"python {preprocess_script} {dataset_folder} {sr_int} 2 ./logs/{model_name} False {percentage}")
else:
    raise FileNotFoundError(f"Script topilmadi: {preprocess_script}")
print("[1/4] Audio qayta ishlandi")

# 2. F0 EXTRACTION
print("[2/4] - F0 chiqarilmoqda...")
if os.path.exists(extract_f0_script):
    ret = os.system(f"python {extract_f0_script} 1 0 0 ./logs/{model_name} False {f0_method}")
else:
    raise FileNotFoundError(f"Script topilmadi: {extract_f0_script}")
print("[2/4] F0 chiqarildi")

# 3. FEATURE EXTRACTION
print("[3/4] - Xususiyatlar chiqarilmoqda...")
if os.path.exists(extract_feature_script):
    ret = os.system(f"python {extract_feature_script} 1 0 ./logs/{model_name} v2 False")
else:
    raise FileNotFoundError(f"Script topilmadi: {extract_feature_script}")
print("[3/4] Xususiyatlar chiqarildi")

# 4. INDEX EXTRACTION
print("[4/4] - Index chiqarilmoqda...")
if os.path.exists(extract_index_script):
    ret = os.system(f"python {extract_index_script} ./logs/{model_name} v2 {index_algorithm}")
else:
    raise FileNotFoundError(f"Script topilmadi: {extract_index_script}")
print("[4/4] Index chiqarildi")

# BATCH SIZE HISOBLASH
try:
    processed_files = [f for f in os.listdir(f"./logs/{model_name}/1_16k_wavs") if f.endswith(".wav")]
    num_processed = len(processed_files)
    recommended_batch_size = max(2, min(32, (num_processed // 80) // 2 * 2))
    print(f"\nQayta ishlangan fragmentlar: {num_processed}")
    print(f"Tavsiya etilgan batch_size: {recommended_batch_size}")
except:
    recommended_batch_size = 8

print("\n\033[92m Malumotlar qayta ishlash tugadi! 3-qadamga oting.\033[0m")
Button(description=" 2-qadam tayyor!", button_style="success")


In [ ]:
#@title <big>&#128994; **3-QADAM: Model treningi (Training)**</big>
#@markdown ### Trening sozlamalari
model_name = "MyVoice" #@param {type:"string"}
sample_rate = "40k" #@param ["32k", "40k", "48k"]
epochs = 500 #@param {type:"integer"}
save_epoch = 50 #@param {type:"integer"}
batch_size = 8 #@param {type:"integer"}
pretrain = "Default" #@param ["Default", "Snowie v3.1", "TITAN-Medium", "KLM v4.3 x3"]
custom_pretrained = False #@param {type:"boolean"}
g_pretrained_link = "" #@param {type:"string"}
d_pretrained_link = "" #@param {type:"string"}
optimizer = "AdamW" #@param ["AdamW", "RAdam"]
fp16_run = True #@param {type:"boolean"}
log_interval = 10 #@param {type:"integer"}
tensorboard = False #@param {type:"boolean"}

ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"

import os
import json
import pathlib
import traceback
from random import shuffle
from urllib.parse import urlparse
from IPython.display import clear_output
from subprocess import PIPE, STDOUT, Popen

os.chdir(ROOT_DIR)
now_dir = os.getcwd()

# PRETRAIN YUKLAB OLISH
param_aria = "--console-log-level=error -c -x 16 -s 16 -k 1M"
hugg_pret = "ht" + "tps:/" + "/hug" + "gin" + "gfa" + "ce.co" + "/Poli" + "tree" + "s/RV" + "C_res" + "ourc" + "es/re" + "solv" + "e/ma" + "in/pret" + "rain" + "ed/v2"

pretrain_outpath = "/content/pretrained_models"
os.system(f"rm -rf {pretrain_outpath} &> /dev/null")
os.makedirs(pretrain_outpath, exist_ok=True)

models = {
    "Default": [
        (f"{sample_rate}/Default/f0D{sample_rate}.pth", "default_D.pth"),
        (f"{sample_rate}/Default/f0G{sample_rate}.pth", "default_G.pth"),
    ],
    "Snowie v3.1": [
        (f"{sample_rate}/Snowie/D_SnowieV3.1_{sample_rate}.pth", "Snowie_D.pth"),
        (f"{sample_rate}/Snowie/G_SnowieV3.1_{sample_rate}.pth", "Snowie_G.pth"),
    ],
    "TITAN-Medium": [
        (f"{sample_rate}/TITAN/D-f0{sample_rate}-TITAN-Medium.pth", "TITAN_Medium_D.pth"),
        (f"{sample_rate}/TITAN/G-f0{sample_rate}-TITAN-Medium.pth", "TITAN_Medium_G.pth"),
    ],
    "KLM v4.3 x3": [
        (f"{sample_rate}/KLM/D_KLM43_X3_{sample_rate}.pth", "KLM_D.pth"),
        (f"{sample_rate}/KLM/G_KLM43_X3_{sample_rate}.pth", "KLM_G.pth"),
    ],
}

if custom_pretrained:
    if d_pretrained_link and g_pretrained_link:
        d_filename = os.path.basename(urlparse(d_pretrained_link).path)
        g_filename = os.path.basename(urlparse(g_pretrained_link).path)
        G_file = f"{pretrain_outpath}/{g_filename}"
        D_file = f"{pretrain_outpath}/{d_filename}"
        print(f"Custom pretrain yuklanmoqda...\nG: {g_filename}\nD: {d_filename}")
        os.system(f"aria2c {param_aria} {g_pretrained_link} -d {pretrain_outpath} -o {g_filename} &> /dev/null")
        os.system(f"aria2c {param_aria} {d_pretrained_link} -d {pretrain_outpath} -o {d_filename} &> /dev/null")
    else:
        raise ValueError("Custom pretrained uchun D va G linklar kiritilmagan!")
else:
    print(f"Pretrain yuklanmoqda: {pretrain}...")
    for f in models[pretrain]:
        os.system(f"aria2c {param_aria} {hugg_pret}/{f[0]} -d {pretrain_outpath} -o {f[1]} &> /dev/null")
    G_file = f"{pretrain_outpath}/{models[pretrain][1][1]}"
    D_file = f"{pretrain_outpath}/{models[pretrain][0][1]}"

print(f"Pretrain tayyor: G={os.path.basename(G_file)}, D={os.path.basename(D_file)}")

# TRENING FUNKSIYASI
def click_train(experiment_dir, sample_rate, save_epoch_interval, total_epochs,
                batch_size, pretrained_G, pretrained_D, optimizer):
    exp_dir = f"{now_dir}/logs/{experiment_dir}"
    os.makedirs(exp_dir, exist_ok=True)
    f0_dir      = f"{exp_dir}/2a_f0"
    f0nsf_dir   = f"{exp_dir}/2b-f0nsf"
    gt_wavs_dir = f"{exp_dir}/0_gt_wavs"
    feature_dir = f"{exp_dir}/3_feature768"
    names = (
        set([name.split(".")[0] for name in os.listdir(gt_wavs_dir)])
        & set([name.split(".")[0] for name in os.listdir(feature_dir)])
        & set([name.split(".")[0] for name in os.listdir(f0_dir)])
        & set([name.split(".")[0] for name in os.listdir(f0nsf_dir)])
    )
    opt = []
    for name in names:
        opt.append(
            f"{gt_wavs_dir}/{name}.wav|"
            f"{feature_dir}/{name}.npy|"
            f"{f0_dir}/{name}.wav.npy|"
            f"{f0nsf_dir}/{name}.wav.npy|0"
        )
    for _ in range(2):
        opt.append(
            f"{now_dir}/logs/mute/0_gt_wavs/mute{sample_rate}.wav|"
            f"{now_dir}/logs/mute/3_feature768/mute.npy|"
            f"{now_dir}/logs/mute/2a_f0/mute.wav.npy|"
            f"{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|0"
        )
    shuffle(opt)
    with open(f"{exp_dir}/filelist.txt", "w") as f:
        f.write("\n".join(opt))
    if sample_rate == "40k":
        config_path = f"configs/v1/{sample_rate}.json"
    else:
        config_path = f"configs/v2/{sample_rate}.json"
    config_save_path = os.path.join(exp_dir, "config.json")
    if not pathlib.Path(config_save_path).exists():
        with open(config_save_path, "w", encoding="utf-8") as f:
            with open(config_path, "r") as config_file:
                config_data = json.load(config_file)
                config_data["train"]["fp16_run"] = fp16_run
                config_data["train"]["log_interval"] = log_interval
                json.dump(config_data, f, ensure_ascii=False, indent=4, sort_keys=True)
            f.write("\n")
    print("\nTrening boshlanmoqda...\n")
    cmd = (
        f"python {ROOT_DIR}/infer/modules/train/train.py "
        f"-e '{experiment_dir}' "
        f"-sr {sample_rate} "
        f"-bs {batch_size} "
        f"-te {total_epochs} "
        f"-se {save_epoch_interval} "
        f"{'-pg %s' % pretrained_G if pretrained_G != '' else ''} "
        f"{'-pd %s' % pretrained_D if pretrained_D != '' else ''} "
        f"-o {optimizer}"
    )
    try:
        p = Popen(cmd, shell=True, cwd=now_dir, stdout=PIPE, stderr=STDOUT,
                  bufsize=1, universal_newlines=True)
        for line in p.stdout:
            print(line.strip())
        p.wait()
    except Exception as e:
        with open(f"{exp_dir}/error_log.txt", "w") as ef:
            ef.write("Trening xatosi:\n")
            ef.write(traceback.format_exc())
        raise Exception(f"Trening xatosi: {e}")
    return "Trening muvaffaqiyatli tugadi!"

if tensorboard:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", "--logdir ./logs --port=8888")

training_log = click_train(
    model_name, sample_rate, save_epoch, epochs,
    batch_size, G_file, D_file, optimizer
)
print(training_log)


In [ ]:
#@title <big>&#128230; **4-QADAM: ZIP yuklab olish (.pth + .index)**</big>
#@markdown ### Trening tugagandan song shu katakni ishga tushiring
model_name = "MyVoice" #@param {type:"string"}

ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"

import os
import glob
import zipfile
from google.colab import files

os.chdir(ROOT_DIR)

print(f"Fayl qidirilmoqda: {model_name}...")

# .pth TOPISH
pth_files = []
weights_dir = f"{ROOT_DIR}/assets/weights"
if os.path.isdir(weights_dir):
    pth_files += glob.glob(f"{weights_dir}/{model_name}.pth")
    pth_files += glob.glob(f"{weights_dir}/{model_name}_e*.pth")
logs_dir = f"{ROOT_DIR}/logs/{model_name}"
if os.path.isdir(logs_dir):
    pth_files += glob.glob(f"{logs_dir}/**/*.pth", recursive=True)

# .index TOPISH
index_files = []
if os.path.isdir(logs_dir):
    index_files += glob.glob(f"{logs_dir}/added_{model_name}_v2.index")
    index_files += glob.glob(f"{logs_dir}/added_*.index")
    index_files += glob.glob(f"{logs_dir}/*.index")
# Takrorlanganlarni ochirib tashlash
index_files = list(dict.fromkeys(index_files))
pth_files = list(dict.fromkeys(pth_files))

print(f"\nTopilgan .pth fayllar ({len(pth_files)} ta):")
for f in pth_files:
    print(f"  {os.path.basename(f)}")
print(f"\nTopilgan .index fayllar ({len(index_files)} ta):")
for f in index_files:
    print(f"  {os.path.basename(f)}")

if not pth_files and not index_files:
    print("\n[XATO] Fayllar topilmadi! Avval 3-qadamni (trening) ishga tushiring.")
    print(f"Qidirilgan yollar:")
    print(f"  - {weights_dir}")
    print(f"  - {logs_dir}")
else:
    main_pth = None
    if pth_files:
        exact = [f for f in pth_files if os.path.basename(f) == f"{model_name}.pth"]
        main_pth = exact[0] if exact else sorted(pth_files)[-1]
    main_index = index_files[0] if index_files else None

    zip_path = f"/content/{model_name}_model.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        if main_pth:
            zf.write(main_pth, os.path.basename(main_pth))
            print(f"\nZIP ga qoshildi: {os.path.basename(main_pth)}")
        if main_index:
            zf.write(main_index, os.path.basename(main_index))
            print(f"ZIP ga qoshildi: {os.path.basename(main_index)}")

    zip_size = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\nZIP yaratildi: {model_name}_model.zip ({zip_size:.1f} MB)")
    print("Yuklab olinmoqda...")
    files.download(zip_path)
    print("\n\033[92m Muvaffaqiyatli! Model ZIP fayl sifatida yuklab olindi.\033[0m")
